In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules
import zipfile

In [2]:
# ==========================================
# 1. CARGA DE DATOS (Optimizada)
# ==========================================
print("Cargando datos...")

# Solo necesitamos orders y order_details para esto
try:
    # Usamos z.open para leer sin descomprimir
    products = pd.read_csv('../data/products.csv.zip')
    order_details = pd.read_csv('../data/order_products__prior.csv.zip')
    print(f"Transacciones cargadas: {len(order_details):,}")
except FileNotFoundError:
    print("Error: Verifica la ruta de los archivos zip en '../data/'")

Cargando datos...
Transacciones cargadas: 32,434,489


In [3]:
# ==========================================
# 2. PREPARACIÓN DE LA "CESTA DE COMPRA" (BASKET)
# ==========================================
# Para FP-Growth, necesitamos una matriz donde:
# Filas = Órdenes (Tickets)
# Columnas = Productos
# Valor = True/False (¿Compró el producto?)

# --- FILTRO DE BIG DATA ---
# Para que la memoria RAM aguante, nos enfocaremos en los productos "Top Sellers".
# Analizaremos TODAS las órdenes, pero solo buscando relaciones entre los 100 productos más populares.
TOP_N_PRODUCTS = 100

print(f"Identificando los {TOP_N_PRODUCTS} productos más vendidos...")
top_products_counts = order_details['product_id'].value_counts().head(TOP_N_PRODUCTS)
top_product_ids = top_products_counts.index.tolist()

# Filtramos el dataset para quedarnos solo con transacciones de estos productos
df_filtered = order_details[order_details['product_id'].isin(top_product_ids)]

print(f"Registros después del filtrado (solo Top {TOP_N_PRODUCTS}): {len(df_filtered):,}")

# Hacemos el cruce con nombres de productos para que sea legible
df_merged = pd.merge(df_filtered, products[['product_id', 'product_name']], on='product_id')

print("Transformando a formato 'Cesta' (One-Hot Encoding)... Esto puede tardar unos segundos.")

# Creamos la matriz Transacción x Producto
# Pivot table: Índice=Order_id, Columnas=Nombre_Producto, Valor=1 si existe
basket = (df_merged.groupby(['order_id', 'product_name'])['product_id']
          .count().unstack().reset_index().fillna(0)
          .set_index('order_id'))

# Convertimos a booleano (True/False) para ahorrar memoria y porque FP-Growth lo requiere
basket = basket.astype(bool)

print(f"Matriz de Cesta lista: {basket.shape} (Órdenes x Productos)")

Identificando los 100 productos más vendidos...
Registros después del filtrado (solo Top 100): 7,483,881
Transformando a formato 'Cesta' (One-Hot Encoding)... Esto puede tardar unos segundos.
Matriz de Cesta lista: (2351240, 100) (Órdenes x Productos)


In [4]:
# ==========================================
# 3. APLICACIÓN DE FP-GROWTH (Unidad 2)
# ==========================================
# Buscamos "Itemsets Frecuentes": grupos de productos que aparecen juntos frecuentemente.
# min_support=0.01 significa que el producto/grupo debe aparecer en al menos el 1% de las órdenes.
# Con 3 millones de órdenes, 1% son 30,000 ventas. Es un umbral alto y sólido.

print("Ejecutando algoritmo FP-Growth...")
frequent_itemsets = fpgrowth(basket, min_support=0.01, use_colnames=True)

print(f"¡Éxito! Se encontraron {len(frequent_itemsets)} conjuntos de items frecuentes.")
print(frequent_itemsets.sort_values(by='support', ascending=False).head(5))

Ejecutando algoritmo FP-Growth...
¡Éxito! Se encontraron 130 conjuntos de items frecuentes.
     support                  itemsets
10  0.200985                  (Banana)
5   0.161383  (Bag of Organic Bananas)
11  0.112572    (Organic Strawberries)
2   0.102891    (Organic Baby Spinach)
6   0.090839    (Organic Hass Avocado)


In [5]:
# ==========================================
# 4. GENERACIÓN DE REGLAS DE ASOCIACIÓN
# ==========================================
# Ahora calculamos las reglas: "Si compra A -> entonces compra B"
# metric="lift": El Lift nos dice qué tan fuerte es la asociación comparado con el azar.
# Lift > 1 significa que hay una relación positiva real.

print("Generando Reglas de Asociación...")
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

# Ordenamos por confianza (probabilidad de que ocurra el consecuente dado el antecedente)
rules = rules.sort_values(by='confidence', ascending=False)

print(f"Reglas encontradas: {len(rules)}")

Generando Reglas de Asociación...
Reglas encontradas: 48


In [6]:
# ==========================================
# 5. ANÁLISIS DE RESULTADOS (Estrategia de Ventas)
# ==========================================

print("\n--- TOP 5 ESTRATEGIAS DE BUNDLING (PAQUETES) ---")
# Mostramos las reglas más fuertes
for i, row in rules.head(5).iterrows():
    antecedents = list(row['antecedents'])[0]
    consequents = list(row['consequents'])[0]
    support = row['support']
    conf = row['confidence']
    lift = row['lift']
    
    print(f"Regla: Si compra '{antecedents}' -> Recomendar '{consequents}'")
    print(f"   - Confianza: {conf*100:.1f}% (Probabilidad de compra)")
    print(f"   - Lift: {lift:.2f}x (Más probable que el azar)")
    print("-" * 30)


--- TOP 5 ESTRATEGIAS DE BUNDLING (PAQUETES) ---
Regla: Si compra 'Organic Fuji Apple' -> Recomendar 'Banana'
   - Confianza: 37.9% (Probabilidad de compra)
   - Lift: 1.88x (Más probable que el azar)
------------------------------
Regla: Si compra 'Honeycrisp Apple' -> Recomendar 'Banana'
   - Confianza: 35.6% (Probabilidad de compra)
   - Lift: 1.77x (Más probable que el azar)
------------------------------
Regla: Si compra 'Cucumber Kirby' -> Recomendar 'Banana'
   - Confianza: 33.0% (Probabilidad de compra)
   - Lift: 1.64x (Más probable que el azar)
------------------------------
Regla: Si compra 'Organic Avocado' -> Recomendar 'Banana'
   - Confianza: 30.2% (Probabilidad de compra)
   - Lift: 1.50x (Más probable que el azar)
------------------------------
Regla: Si compra 'Seedless Red Grapes' -> Recomendar 'Banana'
   - Confianza: 29.7% (Probabilidad de compra)
   - Lift: 1.48x (Más probable que el azar)
------------------------------


In [7]:
# ==========================================
# 1. DEFINICIÓN DEL ESCENARIO (CARRITO DE USUARIO)
# ==========================================
# Imaginemos un usuario con perfil "Healthy" (Comunidad 0)
# Tiene estos productos en su carrito ahora mismo:
carrito_usuario = ['Bag of Organic Bananas', 'Organic Hass Avocado']

print(f"🛒 Carrito del Usuario: {carrito_usuario}")

🛒 Carrito del Usuario: ['Bag of Organic Bananas', 'Organic Hass Avocado']


In [8]:
# ==========================================
# 2. GENERACIÓN DE CANDIDATOS (Recuperación)
# ==========================================
# Usamos las Reglas de Asociación (Unidad 2) para buscar candidatos.
# Buscamos reglas donde el antecedente esté en el carrito.

print("Buscando candidatos basados en reglas...")

# Filtramos las reglas generadas en el Bloque 2
# Convertimos frozensets a strings para buscar fácil
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: list(x)[0])
rules['consequents_str'] = rules['consequents'].apply(lambda x: list(x)[0])

candidates = rules[rules['antecedents_str'].isin(carrito_usuario)].copy()

# Nos quedamos con los productos sugeridos únicos
unique_candidates = candidates['consequents_str'].unique()

print(f"Candidates encontrados: {len(unique_candidates)}")
print(f"Ejemplos: {unique_candidates[:5]}")

Buscando candidatos basados en reglas...
Candidates encontrados: 7
Ejemplos: ['Bag of Organic Bananas' 'Organic Strawberries' 'Organic Hass Avocado'
 'Organic Baby Spinach' 'Organic Raspberries']


In [9]:
# ==========================================
# 3. INGENIERÍA DE CARACTERÍSTICAS (Feature Engineering)
# ==========================================
# Para ordenar (Rankear), necesitamos métricas. Vamos a construir una tabla de features.
# Feature 1: Confidence (Probabilidad de compra dado el carrito) - Viene de FP-Growth
# Feature 2: Lift (Fuerza de la conexión) - Viene de FP-Growth
# Feature 3: PageRank (Autoridad global del producto) - Viene del Grafo
# Feature 4: Community Match (¿Pertenece al mismo barrio que el carrito?) - Viene de Louvain

ranking_table = pd.DataFrame({'product': unique_candidates})

# --- A. Feature de Reglas (Max Confidence/Lift si hay múltiples reglas) ---
def get_rule_metrics(product):
    # Buscamos reglas que lleven a este producto desde items del carrito
    matches = rules[
        (rules['antecedents_str'].isin(carrito_usuario)) & 
        (rules['consequents_str'] == product)
    ]
    if matches.empty:
        return 0, 0
    return matches['confidence'].max(), matches['lift'].max()

ranking_table[['confidence', 'lift']] = ranking_table['product'].apply(
    lambda x: pd.Series(get_rule_metrics(x))
)

# --- B. Feature de Grafo (PageRank) ---
# Mapeamos el PageRank calculado en el bloque 3
ranking_table['pagerank_score'] = ranking_table['product'].map(pagerank).fillna(0)

# --- C. Feature de Comunidad (Community Match) ---
# Determinamos la comunidad dominante del carrito
comm_ids = [partition.get(item, -1) for item in carrito_usuario]
# Usamos la moda (la comunidad más frecuente en el carrito)
carrito_comm = max(set(comm_ids), key=comm_ids.count) 

# Vemos si el candidato es de la misma comunidad
ranking_table['community_id'] = ranking_table['product'].map(partition).fillna(-1)
ranking_table['is_same_community'] = (ranking_table['community_id'] == carrito_comm).astype(int)

print("\n--- Tabla de Características para LTR (Primeras 5 filas) ---")
print(ranking_table.head())

NameError: name 'pagerank' is not defined

In [ ]:
# ==========================================
# 4. MODELO DE SCORING (LTR Pointwise)
# ==========================================
# En un sistema real usaríamos XGBoost Ranker.
# Para este proyecto, crearemos una "Función de Puntuación" manual (Heurística)
# que simula un modelo lineal aprendido: Score = w1*Conf + w2*Lift + w3*PR + w4*Comm

# Pesos hipotéticos (en un modelo real, estos se aprenden con Machine Learning)
w_conf = 0.4   # La probabilidad histórica pesa mucho
w_lift = 0.3   # La fuerza de atracción pesa
w_pr = 0.1     # La popularidad global ayuda un poco
w_comm = 0.2   # El contexto de comunidad ayuda

# Normalizamos para que las escalas no rompan el modelo
ranking_table['lift_norm'] = ranking_table['lift'] / ranking_table['lift'].max()
ranking_table['pr_norm'] = ranking_table['pagerank_score'] / ranking_table['pagerank_score'].max()

# Calculamos el Score Final
ranking_table['final_score'] = (
    (ranking_table['confidence'] * w_conf) +
    (ranking_table['lift_norm'] * w_lift) +
    (ranking_table['pr_norm'] * w_pr) +
    (ranking_table['is_same_community'] * w_comm)
)

In [ ]:
# ==========================================
# 5. RESULTADO FINAL: EL RANKING
# ==========================================
final_ranking = ranking_table.sort_values(by='final_score', ascending=False)

print("\n" + "="*40)
print("   RECOMENDACIONES ORDENADAS (OUTPUT FINAL)")
print("="*40)
rank = 1
for i, row in final_ranking.head(10).iterrows():
    comm_tag = "✅ Misma Comunidad" if row['is_same_community'] else "❌ Otra Comunidad"
    print(f"#{rank} {row['product']}")
    print(f"   Score: {row['final_score']:.4f} | Conf: {row['confidence']:.2f} | Lift: {row['lift']:.2f}")
    print(f"   {comm_tag} (PR: {row['pagerank_score']:.4f})")
    print("-" * 30)
    rank += 1

# ==========================================
# 6. GRÁFICO DE IMPORTANCIA
# ==========================================
plt.figure(figsize=(10, 6))
# Top 10 productos
top_10 = final_ranking.head(10)
plt.barh(top_10['product'][::-1], top_10['final_score'][::-1], color='teal')
plt.xlabel('Score de Relevancia (LTR)')
plt.title(f'Top 10 Recomendaciones para: {carrito_usuario}')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()